# Preprocessing

In [2]:
import pandas as pd

data_path = "../01_data/DataCoSupplyChainDataset.csv"

raw_df = pd.read_csv(data_path, encoding="latin-1")

display(raw_df.shape)
display(raw_df.info())
raw_df.head()

(180519, 53)

<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname                 180519 non

None

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [3]:
# Check for datetime
raw_df[
    ["order date (DateOrders)", "shipping date (DateOrders)"]
].head(10)

,order date (DateOrders),shipping date (DateOrders)
0,1/31/2018 22:56,2/3/2018 22:56
1,1/13/2018 12:27,1/18/2018 12:27
2,1/13/2018 12:06,1/17/2018 12:06
3,1/13/2018 11:45,1/16/2018 11:45
4,1/13/2018 11:24,1/15/2018 11:24
5,1/13/2018 11:03,1/19/2018 11:03
6,1/13/2018 10:42,1/15/2018 10:42
7,1/13/2018 10:21,1/15/2018 10:21
8,1/13/2018 10:00,1/16/2018 10:00
9,1/13/2018 9:39,1/15/2018 9:39


In [4]:
# Create a working copy to preserve the original imported data
clean_df = raw_df.copy()

# Convert order and shipping timestamps from strings to datetime
clean_df["order date (DateOrders)"] = pd.to_datetime(
    clean_df["order date (DateOrders)"],
    format="%m/%d/%Y %H:%M",
)

clean_df["shipping date (DateOrders)"] = pd.to_datetime(
    clean_df["shipping date (DateOrders)"],
    format="%m/%d/%Y %H:%M",
)

clean_df[
    ["order date (DateOrders)", "shipping date (DateOrders)"]
].dtypes

order date (DateOrders)       datetime64[us]
shipping date (DateOrders)    datetime64[us]
dtype: object

In [5]:
# Check for exact duplicate rows
duplicate_rows = raw_df.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows:,}")

Duplicate rows: 0


In [6]:
# Identify missing values across all columns in the cleaned dataset
missing_summary = pd.DataFrame(
    {
        "Missing Count": clean_df.isna().sum(),
        "Missing Percent": (clean_df.isna().mean() * 100).round(2),
    }
)

# Display only columns containing at least one missing value
missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values("Missing Percent", ascending=False)

missing_summary

,Missing Count,Missing Percent
Product Description,180519,100.00
Order Zipcode,155679,86.24
Customer Lname,8,0.00
Customer Zipcode,3,0.00


**Missing-value assessment:** Product Description is entirely missing and provides no usable information. Order Zipcode is populated exclusively for U.S. destination records, indicating structural rather than unexplained missingness. Missing values in Customer Lname (8 records) and Customer Zipcode (3 records) are negligible and do not warrant row removal or imputation at this stage.

In [7]:
# Compare Order Zipcode availability across destination countries
zipcode_by_country = (
    clean_df.groupby("Order Country")["Order Zipcode"]
    .agg(
        total_records="size",
        zipcode_present="count",
    )
)

# Calculate the percentage of records with a populated Order Zipcode
zipcode_by_country["zipcode_present_pct"] = (
    zipcode_by_country["zipcode_present"]
    / zipcode_by_country["total_records"]
    * 100
).round(2)

# Show countries with the most populated Order Zipcode values
zipcode_by_country.sort_values(
    "zipcode_present",
    ascending=False,
).head(20)

,total_records,zipcode_present,zipcode_present_pct
Order Country,,,
Estados Unidos,24840,24840,100.0
Afganistán,163,0,0.0
Papúa Nueva Guinea,61,0,0.0
Nigeria,2309,0,0.0
Noruega,333,0,0.0
Nueva Zelanda,1590,0,0.0
Níger,108,0,0.0
Omán,8,0,0.0
Pakistán,758,0,0.0


Order Zipcode is populated exclusively for U.S. destination records. Missing values for international orders therefore reflect the dataset's U.S.-specific ZIP code field rather than a data-quality issue. No imputations or drops are necessary.

In [8]:
# Drop fields with no meaningful analytical value for delivery-risk modeling
columns_to_drop = [
    "Product Description",  # Entirely missing
    "Product Status",       # Constant value across all records
    "Customer Password",    # Anonymized account information
    "Customer Email",       # Anonymized account information
    "Customer Fname",       # Customer-identifying information
    "Customer Lname",       # Customer-identifying information
    "Customer Street",      # Granular customer-identifying information
    "Latitude",             # Granular location information
    "Longitude",            # Granular location information
    "Product Image",        # Product image URL with no operational relevance
    "Order Item Id",        # Unique line-item identifier
]

clean_df = clean_df.drop(columns=columns_to_drop)

print(f"Remaining columns: {clean_df.shape[1]}")

Remaining columns: 42


In [9]:
# Count unique values in each column
unique_counts = clean_df.nunique().sort_values()

unique_counts

Customer Country                     2
Late_delivery_risk                   2
Customer Segment                     3
Type                                 4
Delivery Status                      4
Shipping Mode                        4
Days for shipment (scheduled)        4
Order Item Quantity                  5
Market                               5
Days for shipping (real)             7
Order Status                         9
Department Id                       11
Department Name                     11
Order Item Discount Rate            18
Order Region                        23
Customer State                      46
Category Name                       50
Category Id                         51
Product Category Id                 51
Product Price                       75
Order Item Product Price            75
Product Card Id                    118
Order Item Cardprod Id             118
Product Name                       118
Order Item Profit Ratio            162
Order Country            

In [10]:
# Identify columns that contain exactly the same values across all records
duplicate_columns = []

columns = clean_df.columns

for i, col_a in enumerate(columns):
    for col_b in columns[i + 1:]:
        if clean_df[col_a].equals(clean_df[col_b]):
            duplicate_columns.append((col_a, col_b))

# Display any exact duplicate column pairs found
duplicate_columns

[('Benefit per order', 'Order Profit Per Order'),
 ('Sales per customer', 'Order Item Total'),
 ('Category Id', 'Product Category Id'),
 ('Customer Id', 'Order Customer Id'),
 ('Order Item Cardprod Id', 'Product Card Id'),
 ('Order Item Product Price', 'Product Price')]

In [11]:
# Drop columns that are exact duplicates of retained fields
duplicate_columns_to_drop = [
    "Benefit per order",           # Identical to Order Profit Per Order
    "Sales per customer",          # Identical to Order Item Total
    "Product Category Id",         # Identical to Category Id
    "Order Customer Id",           # Identical to Customer Id
    "Order Item Cardprod Id",      # Identical to Product Card Id
    "Order Item Product Price",    # Identical to Product Price
]

clean_df = clean_df.drop(columns=duplicate_columns_to_drop)

print(f"Remaining columns: {clean_df.shape[1]}")

Remaining columns: 36


In [12]:
# Define ID and descriptive label pairs to test for one-to-one mappings
mapping_pairs = [
    ("Department Id", "Department Name"),
    ("Category Id", "Category Name"),
    ("Product Card Id", "Product Name"),
]

# Check the maximum number of unique values in both mapping directions
for id_col, label_col in mapping_pairs:
    id_to_label = clean_df.groupby(id_col)[label_col].nunique().max()
    label_to_id = clean_df.groupby(label_col)[id_col].nunique().max()

    print(f"{id_col} -> {label_col}: {id_to_label}")
    print(f"{label_col} -> {id_col}: {label_to_id}")
    print()

Department Id -> Department Name: 1
Department Name -> Department Id: 1

Category Id -> Category Name: 1
Category Name -> Category Id: 2

Product Card Id -> Product Name: 1
Product Name -> Product Card Id: 1



In [13]:
# Drop ID fields that map one-to-one with retained descriptive labels
redundant_id_columns = [
    "Department Id",      # One-to-one with Department Name
    "Product Card Id",    # One-to-one with Product Name
]

clean_df = clean_df.drop(columns=redundant_id_columns)
clean_df.shape

(180519, 34)

In [14]:
# Identify category names associated with more than one Category ID
category_id_counts = clean_df.groupby("Category Name")[
    "Category Id"
].nunique()

duplicate_category_names = category_id_counts[
    category_id_counts > 1
].index

# Inspect duplicated category names and their department assignments
clean_df.loc[
    clean_df["Category Name"].isin(duplicate_category_names),
    ["Category Id", "Category Name", "Department Name"],
].drop_duplicates().sort_values(
    ["Category Name", "Category Id"]
)

,Category Id,Category Name,Department Name
55,13,Electronics,Footwear
62,37,Electronics,Outdoors


In [15]:
# Check how many categories and departments each product maps to
product_mapping = (
    clean_df.groupby("Product Name")
    .agg(
        categories=("Category Name", "nunique"),
        departments=("Department Name", "nunique"),
    )
)

# Display the maximum number of categories and departments per product
product_mapping.max()

categories     1
departments    1
dtype: int64

In [16]:
# Check whether each Category Id maps to only one department
clean_df.groupby("Category Id")["Department Name"].nunique().max()

np.int64(1)

**Product hierarchy:** Each product maps uniquely to one category and one department. Therefore, category and department fields are not separately encoded in the order-level dataset because their information is already represented by the retained product features. Category and department classifications can be recovered from the product mapping if needed for later analysis or interpretation.

In [17]:
# Check how many unique values each column can have within a single order
within_order_variation = {}

for column in clean_df.columns:
    if column != "Order Id":
        max_unique = (
            clean_df.groupby("Order Id")[column]
            .nunique(dropna=False)
            .max()
        )
        within_order_variation[column] = max_unique

# Sort columns from least to most within-order variation
within_order_variation = pd.Series(
    within_order_variation,
    name="Max Unique Values Within Order",
).sort_values()

within_order_variation

Type                             1
Order Zipcode                    1
Order Status                     1
Order State                      1
Order Region                     1
order date (DateOrders)          1
shipping date (DateOrders)       1
Order City                       1
Market                           1
Customer Zipcode                 1
Customer State                   1
Order Country                    1
Customer Id                      1
Days for shipping (real)         1
Days for shipment (scheduled)    1
Delivery Status                  1
Late_delivery_risk               1
Customer Segment                 1
Shipping Mode                    1
Customer City                    1
Customer Country                 1
Order Item Discount Rate         5
Order Item Profit Ratio          5
Order Item Quantity              5
Category Id                      5
Order Item Total                 5
Order Profit Per Order           5
Category Name                    5
Department Name     

In [18]:
# Identify fields that contain only one unique value within every order
order_level_columns = [
    column
    for column in clean_df.columns
    if column != "Order Id"
    and clean_df.groupby("Order Id")[column]
    .nunique(dropna=False)
    .max()
    == 1
]

order_level_columns

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Delivery Status',
 'Late_delivery_risk',
 'Customer City',
 'Customer Country',
 'Customer Id',
 'Customer Segment',
 'Customer State',
 'Customer Zipcode',
 'Market',
 'Order City',
 'Order Country',
 'order date (DateOrders)',
 'Order Region',
 'Order State',
 'Order Status',
 'Order Zipcode',
 'shipping date (DateOrders)',
 'Shipping Mode']

In [19]:
# Retain one record per order for fields that are constant within orders
order_level_df = (
    clean_df[["Order Id"] + order_level_columns]
    .drop_duplicates(subset="Order Id")
    .set_index("Order Id")
)

# Confirm the dimensions of the order-level dataframe
order_level_df.shape

(65752, 21)

In [20]:
# Aggregate product quantities within each order, summing quantities when
# the same product appears on multiple line-item records
product_quantities = clean_df.pivot_table(
    index="Order Id",
    columns="Product Name",
    values="Order Item Quantity",
    aggfunc="sum",
    fill_value=0,
)

# Add a prefix to distinguish engineered product-quantity features
product_quantities = product_quantities.add_prefix("Product Qty - ")

# Confirm the dimensions of the product-quantity matrix
product_quantities.shape

(65752, 118)

In [21]:
# Combine order-level attributes with product-level quantity features
order_df = order_level_df.join(product_quantities)

# Restore Order Id as a standard column
order_df = order_df.reset_index()

# Confirm the dimensions of the order-level dataframe
order_df.shape

(65752, 140)

In [22]:
# Validate that each Order Id appears exactly once
assert order_df["Order Id"].is_unique

# Validate that the number of rows matches the number of unique source orders
assert len(order_df) == clean_df["Order Id"].nunique()

# Validate that total product quantities were preserved during aggregation
product_quantity_columns = [
    column
    for column in order_df.columns
    if column.startswith("Product Qty - ")
]

assert (
    order_df[product_quantity_columns].to_numpy().sum()
    == clean_df["Order Item Quantity"].sum()
)

print("Order-level aggregation validated successfully.")

Order-level aggregation validated successfully.


In [23]:
# Check whether gross sales equal product price multiplied by quantity
sales_match = (
    clean_df["Sales"].round(2)
    == (
        clean_df["Product Price"]
        * clean_df["Order Item Quantity"]
    ).round(2)
).all()

sales_match

np.True_

In [24]:
# Check whether net item totals equal gross sales minus discounts
item_total_match = (
    clean_df["Order Item Total"].round(2)
    == (
        clean_df["Sales"]
        - clean_df["Order Item Discount"]
    ).round(2)
).all()

item_total_match

np.False_

In [25]:
# Calculate the difference between the recorded total and the expected
# total based on gross sales minus the recorded discount
item_total_difference = (
    clean_df["Order Item Total"]
    - (
        clean_df["Sales"]
        - clean_df["Order Item Discount"]
    )
)

# Examine the distribution of the discrepancies
item_total_difference.describe()

count    180519.000000
mean          0.000254
std           0.001577
min          -0.000042
25%          -0.000004
50%           0.000000
75%           0.000000
max           0.010014
dtype: float64

**Financial field validation:** Gross sales are consistent with product price multiplied by item quantity across all records. Net item totals are also consistent with gross sales minus discounts, with discrepancies of no more than approximately $0.01 due to the source data's intermediate rounding precision. To preserve the original financial values, order-level financial features are therefore calculated by summing the recorded line-item sales, discounts, and net totals rather than recalculating net sales from the aggregated components.

In [26]:
# Add aggregated financial values to the order-level dataset
order_level_df["Gross Order Sales"] = (
    clean_df.groupby("Order Id")["Sales"].sum().round(2)
)

order_level_df["Total Order Discount"] = (
    clean_df.groupby("Order Id")["Order Item Discount"].sum().round(2)
)

order_level_df["Net Order Sales"] = (
    clean_df.groupby("Order Id")["Order Item Total"].sum().round(2)
)

# Calculate the effective discount rate for each order
order_level_df["Order Discount Rate"] = (
    order_level_df["Total Order Discount"]
    / order_level_df["Gross Order Sales"]
).round(4)

# Combine order-level attributes with product quantity features
order_df = order_level_df.join(product_quantities).reset_index()

# Confirm the dimensions of the order-level dataframe
order_df.shape

(65752, 144)

In [27]:
# Calculate the profit ratio implied by recorded profit and net sales
calculated_profit_ratio = (
    clean_df["Order Profit Per Order"]
    / clean_df["Order Item Total"]
)

# Compare the calculated ratio with the recorded profit ratio
profit_ratio_difference = (
    calculated_profit_ratio
    - clean_df["Order Item Profit Ratio"]
)

profit_ratio_difference.abs().max().round(3)

np.float64(0.006)

In [28]:
# Derive the implied cost represented by each line item
clean_df["Implied Line Cost"] = (
    clean_df["Order Item Total"]
    - clean_df["Order Profit Per Order"]
)

# Convert line cost to an implied unit cost
clean_df["Implied Unit Cost"] = (
    clean_df["Implied Line Cost"]
    / clean_df["Order Item Quantity"]
)

clean_df.groupby("Product Name")["Implied Unit Cost"].agg(
    ["count", "min", "median", "max"]
)

,count,min,median,max
Product Name,,,,
Adult dog supplies,492,32.279999,57.749999,292.449997
Baby sweater,207,23.040001,39.950001,190.180000
Bag Boy Beverage Holder,279,9.745000,16.870000,77.670003
Bag Boy M330 Push Cart,69,33.277500,51.826665,293.962494
Bowflex SelectTech 1090 Dumbbells,10,274.489990,403.789993,579.189991
...,...,...,...,...
adidas Kids' F5 Messi FG Soccer Cleat,262,13.910000,22.870418,123.443331
adidas Men's F10 Messi TRX FG Soccer Cleat,305,22.946665,39.950002,195.719997
adidas Men's Germany Black Crest Away Tee,289,9.560000,16.788000,84.380001


In [29]:
# Aggregate recorded line-item profit to the order level
order_df["Total Order Profit"] = (
    clean_df.groupby("Order Id")["Order Profit Per Order"]
    .sum()
    .round(2)
)

# Calculate the effective profit margin for each order
order_df["Order Profit Margin"] = (
    order_df["Total Order Profit"]
    / order_df["Net Order Sales"]
).round(4)

# Confirm the dimensions of the order-level dataframe
order_df.shape

(65752, 146)

**Profit aggregation:** Order Profit Per Order varies across line items despite its name. The recorded Order Item Profit Ratio was validated against line-item profit divided by net item sales, with differences consistent with rounding. Therefore, recorded profit was summed across line items to obtain total order profit, and an order-level profit margin was calculated from total order profit divided by net order sales.

Initial preprocessing identified 180,519 line-item records representing 65,752 unique orders from 20,652 customers. Delivery risk is constant within each order, confirming that the appropriate analytical unit for the late-delivery problem is the order rather than the line item. Obvious null, irrelevant, and redundant fields were dropped, and remaining attributes were evaluated to determine whether they operate at the order or line-item level.

To preserve line-item information without duplicating order outcomes, an order-level dataset was created retaining order attributes, aggregating financial measures, and pivoting the 118 products into quantity features. Product mapping was also validated and showed that each product maps uniquely to one category and one department, so separate category and department features would be redundant and were not retained. The resulting dataset contains one observation per order while preserving the products and quantities ordered and relevant order-level financial information.

In [30]:
# Export the order-level baseline dataset for downstream analysis
order_df.to_csv(
    "../01_data/DataCo_order_level.csv",
    index=False,
)

In [31]:
# Confirm the final dimensions of the exported baseline dataset
print(f"Final order-level dataset: {order_df.shape[0]:,} rows × {order_df.shape[1]:,} columns")

Final order-level dataset: 65,752 rows × 146 columns
